# Knowledge Engineering: Autonomous Trailer Agent

**ICS 3202: Artificial Intelligence, Practical Exercise 4**

| Name | Admission No. |
|---|---|
| Laureen | 191876 |
| Moses | 191158 |
| Vanessa | 190378 |

In Practical Exercise 3 we designed the self-driving trailer as a **searching agent**: we built a directed graph of Kenya's highway network and used path costs (km) to find the optimal route from Nairobi to Kisumu.

In this exercise we turn the trailer into a **knowledge-based agent**. It keeps a **knowledge base (KB)** of facts and rules about the roads, signals and road conditions. It **adds new facts** as it learns about the world, and **queries** the KB to decide what to do and which route to take. We follow the knowledge engineering process step by step.

## Step 1: Identify the Task

The task is to help a **self-driving trailer** travel safely and efficiently on Kenya's highway network, from **Nairobi** to **Kisumu**.

The agent must answer two kinds of questions:

1. **Route questions:** *Which route should I take from where I am to my destination?* The answer is the optimal (least-distance) path that avoids closed roads.
2. **Action questions:** *What should I do right now?* The answer depends on the traffic signal, the weather and the condition of the road ahead.

**Goal:** make safe driving decisions with minimal risk, while reaching the destination using the shortest available route.

## Step 2: Assemble Relevant Knowledge

The agent needs to know:

| Knowledge | Source |
|---|---|
| Towns on the route and the highways connecting them | Class map of Kenya's Highway Network and the Google Maps view (Practical Exercise 3) |
| Road distance between connected towns (path cost) | Google Maps distance search (Practical Exercise 3, `design.md` in our **PS-Agents** repo) |
| What to do at each traffic signal | Kenyan traffic rules (Highway Code) |
| How to respond to weather and road conditions (fog, rain, accidents, road works) | Road safety guidelines and our PEAS environment analysis (Practical Exercise 3) |
| What to do at checkpoints (weighbridges, police checks) | Kenyan traffic rules for heavy commercial vehicles |

**APIs / web scraping:** none. Distances were collected manually using the Google Maps search feature and recorded in Practical Exercise 3. The knowledge was abstracted to the main junction towns between Nairobi and Kisumu; towns east of Nairobi (Nyeri, Meru, Embu, Thika) were left out because they do not lead towards Kisumu.

## Step 3: Define the Vocabulary

**Predicates** (represented in Python as dictionaries, keys and functions):

| Predicate | Meaning | Python representation |
|---|---|---|
| `Town(x)` | x is a town in the network | the keys of `roads` |
| `Road(x, y, d)` | there is a highway from x to y, d km long | `roads[x][y] = d` |
| `Signal(s)` | the trailer observes traffic signal s | key of `signal_rules` |
| `Condition(x, y, c)` | the road from x to y currently has condition c | `road_conditions[(x, y)] = c` |
| `Closed(x, y)` | the road from x to y cannot be used | a condition whose rule closes the road |
| `CanDrive(x, y)` | `Road(x, y, d)` exists and the road is **not** `Closed(x, y)` | `can_drive(x, y)` |
| `Action(s)` | the action the agent should take | value returned by `decide_action()` |

**Constants:**

- **Towns:** Nairobi, Naivasha, Nakuru, Molo, Kericho, Narok, Bomet, Kisii, Eldoret, Kapsabet, Kakamega, Kisumu
- **Signals:** Red, Yellow, Green, Flashing Red, Flashing Yellow, Off
- **Conditions:** Clear, Fog, Heavy Rain, Accident, Road Works, Flooded, Weighbridge, Police Check

## Step 4: Encode General Knowledge about the Domain

### 4.1 The highway network (from Practical Exercise 3)

The directed graph from Practical Exercise 3 is stored as an **adjacency dictionary**. Each town maps to the towns it connects to, with the distance in km. Every entry is a fact of the form `Road(x, y, d)`.

In [1]:
# Road(x, y, d): there is a highway from town x to town y, d km long
# Distances from Google Maps (Practical Exercise 3)
roads = {
    "Nairobi":  {"Naivasha": 91, "Narok": 143},
    "Naivasha": {"Nakuru": 71},
    "Nakuru":   {"Molo": 51, "Eldoret": 156},
    "Molo":     {"Kericho": 64},
    "Narok":    {"Bomet": 81},
    "Bomet":    {"Kisii": 82, "Kericho": 76},
    "Eldoret":  {"Kapsabet": 46, "Kakamega": 106},
    "Kericho":  {"Kisumu": 84},
    "Kisii":    {"Kisumu": 107},
    "Kapsabet": {"Kisumu": 97},
    "Kakamega": {"Kisumu": 53},
    "Kisumu":   {},          # goal town: no outgoing roads needed
}

print("Towns in the KB:", len(roads))
print("Roads in the KB:", sum(len(n) for n in roads.values()))

Towns in the KB: 12
Roads in the KB: 15


### 4.2 Rules

General rules tell the agent what to do in each situation:

- **Signal rules:** `Signal(s) ⇒ Action(a)`
- **Condition rules:** `Condition(x, y, c) ⇒ Action(a)`. Some conditions also make the road unusable: `Condition(x, y, Accident) ⇒ Closed(x, y)`.

In [2]:
# Signal(s) => Action(a)
signal_rules = {
    "Red":             "Stop: it is unsafe to move.",
    "Yellow":          "Slow down: prepare to stop.",
    "Green":           "Go: it is safe to proceed.",
    "Flashing Red":    "Stop completely, then proceed only when the junction is clear.",
    "Flashing Yellow": "Proceed slowly with caution.",
    "Off":             "Treat as a stop sign: stop, check the junction, then proceed.",
}

# Condition(x, y, c) => Action(a)  [and possibly Closed(x, y)]
# Each condition has: the action to take, and whether it closes the road.
condition_rules = {
    "Clear":        {"action": "Drive normally within the speed limit.",            "closes_road": False},
    "Fog":          {"action": "Slow down, switch on fog lights, keep a long following distance.", "closes_road": False},
    "Heavy Rain":   {"action": "Reduce speed and avoid sudden braking (risk of skidding).", "closes_road": False},
    "Road Works":   {"action": "Slow down and follow diversion signs.",               "closes_road": False},
    "Weighbridge":  {"action": "Stop at the weighbridge to have the load weighed.",   "closes_road": False},
    "Police Check": {"action": "Slow down and stop if signalled by the police.",      "closes_road": False},
    "Accident":     {"action": "Road blocked: re-plan the route.",                    "closes_road": True},
    "Flooded":      {"action": "Road impassable: re-plan the route.",                 "closes_road": True},
}

# Facts about the current state of each road: Condition(x, y, c)
# Empty at the start: every road is assumed Clear until the agent learns otherwise.
road_conditions = {}

# Safety default: what to do when the agent sees something it does not know.
UNKNOWN_SIGNAL_ACTION = "Unknown signal: STOP and wait until it is safe (safety first)."

# Signals after which the trailer must not move yet
STOP_SIGNALS = {"Red", "Flashing Red", "Off"}

### 4.3 Displaying the knowledge base

In [3]:
def show_kb():
    """Print everything the agent currently knows."""
    print("=" * 60)
    print("KNOWLEDGE BASE")
    print("=" * 60)
    print("\nRoads  Road(x, y, d):")
    for town, neighbours in roads.items():
        for nxt, km in neighbours.items():
            print(f"  {town:9} -> {nxt:9} {km:4} km")
    print("\nSignal rules  Signal(s) => Action(a):")
    for s, a in signal_rules.items():
        print(f"  {s:16} -> {a}")
    print("\nCondition rules  Condition(c) => Action(a):")
    for c, r in condition_rules.items():
        closed = "  [CLOSES ROAD]" if r["closes_road"] else ""
        print(f"  {c:13} -> {r['action']}{closed}")
    print("\nCurrent road conditions  Condition(x, y, c):")
    if not road_conditions:
        print("  (none reported: all roads Clear)")
    for (x, y), c in road_conditions.items():
        print(f"  {x} -> {y}: {c}")
    print("=" * 60)

show_kb()

KNOWLEDGE BASE

Roads  Road(x, y, d):
  Nairobi   -> Naivasha    91 km
  Nairobi   -> Narok      143 km
  Naivasha  -> Nakuru      71 km
  Nakuru    -> Molo        51 km
  Nakuru    -> Eldoret    156 km
  Molo      -> Kericho     64 km
  Narok     -> Bomet       81 km
  Bomet     -> Kisii       82 km
  Bomet     -> Kericho     76 km
  Eldoret   -> Kapsabet    46 km
  Eldoret   -> Kakamega   106 km
  Kericho   -> Kisumu      84 km
  Kisii     -> Kisumu     107 km
  Kapsabet  -> Kisumu      97 km
  Kakamega  -> Kisumu      53 km

Signal rules  Signal(s) => Action(a):
  Red              -> Stop: it is unsafe to move.
  Yellow           -> Slow down: prepare to stop.
  Green            -> Go: it is safe to proceed.
  Flashing Red     -> Stop completely, then proceed only when the junction is clear.
  Flashing Yellow  -> Proceed slowly with caution.
  Off              -> Treat as a stop sign: stop, check the junction, then proceed.

Condition rules  Condition(c) => Action(a):
  Clear       

### 4.4 Inference: `CanDrive(x, y)` and route finding

The agent **infers** new knowledge from what it already knows:

- `CanDrive(x, y)` is true when `Road(x, y, d)` exists **and** the road is not `Closed(x, y)`.
- The best route is found with **Uniform Cost Search (UCS)**, the same algorithm we used in Practical Exercise 3. UCS always expands the path with the **lowest total cost g(n)** first, and only stops when the goal is the cheapest path in the frontier, so the path it returns is optimal. Closed roads are skipped, so the agent automatically re-plans when it learns about an accident.

In [4]:
import heapq

def normalise(name):
    """Make input consistent: ' nakuru ' -> 'Nakuru'."""
    return name.strip().title()

def is_closed(x, y):
    """Closed(x, y): true if the road's current condition closes it."""
    condition = road_conditions.get((x, y), "Clear")
    return condition_rules.get(condition, {"closes_road": False})["closes_road"]

def can_drive(x, y):
    """CanDrive(x, y) <= Road(x, y, d) AND NOT Closed(x, y)"""
    return y in roads.get(x, {}) and not is_closed(x, y)

def ucs(start, goal, verbose=False):
    """Uniform Cost Search over the KB. Returns (path, cost) or (None, None)."""
    start, goal = normalise(start), normalise(goal)
    frontier = [(0, [start])]          # priority queue ordered by path cost g(n)
    expanded = set()
    step = 0
    while frontier:
        if verbose:
            step += 1
            listing = ", ".join(f"{'-'.join(p)} (g={g})" for g, p in sorted(frontier))
            print(f"Step {step}: frontier = [{listing}]")
        g, path = heapq.heappop(frontier)   # cheapest path first
        town = path[-1]
        if town == goal:                      # goal test when the path is chosen for expansion
            return path, g
        if town in expanded:                  # already reached more cheaply: discard
            continue
        expanded.add(town)
        for nxt, km in roads.get(town, {}).items():
            if can_drive(town, nxt) and nxt not in expanded:
                heapq.heappush(frontier, (g + km, path + [nxt]))
    return None, None

## Step 5: Add New Facts to the Knowledge Base

A knowledge-based agent must be able to **learn**. These functions add new facts to the KB without changing any code:

- `add_road()` adds a new highway, `Road(x, y, d)`
- `report_condition()` records what is happening on a road, `Condition(x, y, c)`
- `add_signal_rule()` teaches the agent a new signal

Each function checks its input first, so wrong facts (unknown towns, negative distances) do not get into the KB.

In [5]:
def add_road(x, y, km):
    """Add the fact Road(x, y, km) to the KB."""
    x, y = normalise(x), normalise(y)
    if km <= 0:
        print(f"Rejected: distance must be positive (got {km}).")
        return
    roads.setdefault(x, {})
    roads.setdefault(y, {})
    roads[x][y] = km
    print(f"Added fact: Road({x}, {y}, {km} km)")

def report_condition(x, y, condition):
    """Add the fact Condition(x, y, c) to the KB."""
    x, y, condition = normalise(x), normalise(y), normalise(condition)
    if y not in roads.get(x, {}):
        print(f"Rejected: there is no road {x} -> {y} in the KB.")
        return
    if condition not in condition_rules:
        print(f"Rejected: unknown condition '{condition}'. Known: {list(condition_rules)}")
        return
    if condition == "Clear":
        road_conditions.pop((x, y), None)
    else:
        road_conditions[(x, y)] = condition
    closed = " (road is now CLOSED)" if condition_rules[condition]["closes_road"] else ""
    print(f"Added fact: Condition({x}, {y}, {condition}){closed}")

def add_signal_rule(signal, action):
    """Add the rule Signal(s) => Action(a) to the KB."""
    signal = normalise(signal)
    signal_rules[signal] = action
    print(f"Added rule: Signal({signal}) => '{action}'")

# Examples: adding new knowledge
add_signal_rule("Pedestrian Crossing", "Slow down and give way to pedestrians.")
add_road("Kericho", "Kisii", 103)          # a new connection between the B1 and B3 corridors
report_condition("Molo", "Kericho", "Fog") # fog is common on the Molo-Kericho stretch
report_condition("Nakuru", "Eldoret", "Weighbridge")

show_kb()

Added rule: Signal(Pedestrian Crossing) => 'Slow down and give way to pedestrians.'
Added fact: Road(Kericho, Kisii, 103 km)
Added fact: Condition(Molo, Kericho, Fog)
Added fact: Condition(Nakuru, Eldoret, Weighbridge)
KNOWLEDGE BASE

Roads  Road(x, y, d):
  Nairobi   -> Naivasha    91 km
  Nairobi   -> Narok      143 km
  Naivasha  -> Nakuru      71 km
  Nakuru    -> Molo        51 km
  Nakuru    -> Eldoret    156 km
  Molo      -> Kericho     64 km
  Narok     -> Bomet       81 km
  Bomet     -> Kisii       82 km
  Bomet     -> Kericho     76 km
  Eldoret   -> Kapsabet    46 km
  Eldoret   -> Kakamega   106 km
  Kericho   -> Kisumu      84 km
  Kericho   -> Kisii      103 km
  Kisii     -> Kisumu     107 km
  Kapsabet  -> Kisumu      97 km
  Kakamega  -> Kisumu      53 km

Signal rules  Signal(s) => Action(a):
  Red              -> Stop: it is unsafe to move.
  Yellow           -> Slow down: prepare to stop.
  Green            -> Go: it is safe to proceed.
  Flashing Red     -> Stop 

## Steps 6 & 7: Encode a Specific Problem Instance and Pose Queries

### Step 6: The current situation (facts from the trailer's sensors)

The **problem instance** is what the trailer observes right now: where it is, where it is going, and the traffic signal its camera sees.

In [6]:
# Step 6: current observation from GPS and camera
current_state = {
    "location":    "Nairobi",
    "destination": "Kisumu",
    "signal":      "Green",
}

print("Current situation:")
for k, v in current_state.items():
    print(f"  {k:12}: {v}")

Current situation:
  location    : Nairobi
  destination : Kisumu
  signal      : Green


### Step 7: The agent queries the KB

`decide_action()` combines everything in the KB:

1. It looks up the action for the **signal** it sees (unknown signal ⇒ stop, for safety).
2. It asks UCS for the **best route** that avoids closed roads.
3. It checks the **condition of the next road** on that route and adds that advice.
4. It gives a **final decision**: the signal rule comes first (e.g. a red light means wait, even if the road ahead is clear).

In [7]:
def decide_action(state, verbose_search=False):
    """Query the KB: what should the trailer do right now, and which route should it take?"""
    location    = normalise(state["location"])
    destination = normalise(state["destination"])
    signal      = normalise(state.get("signal", "Off"))

    print("\n=== Autonomous Trailer Decision ===")
    print(f"Location: {location} | Destination: {destination} | Signal: {signal}")

    # Check the towns are in the KB
    for town in (location, destination):
        if town not in roads:
            print(f"FINAL DECISION: '{town}' is not in the knowledge base. STOP and request an updated map.")
            return

    # 1. Signal rule
    signal_action = signal_rules.get(signal, UNKNOWN_SIGNAL_ACTION)
    print(f"Signal rule: {signal_action}")

    # 2. Route query (UCS)
    if location == destination:
        print("FINAL DECISION: already at the destination. Park and offload the cargo.")
        return
    path, cost = ucs(location, destination, verbose=verbose_search)
    if path is None:
        print("FINAL DECISION: NO open route to the destination. STOP and wait for roads to reopen.")
        return
    print(f"Best route: {' -> '.join(path)} ({cost} km)")

    # 3. Condition of the next road
    nxt = path[1]
    condition = road_conditions.get((location, nxt), "Clear")
    print(f"Next road: {location} -> {nxt} is {condition}: {condition_rules[condition]['action']}")

    # Warnings for roads further along the route
    for a, b in zip(path[1:], path[2:]):
        c = road_conditions.get((a, b))
        if c:
            print(f"Ahead: {a} -> {b} has {c}: {condition_rules[c]['action']}")

    # 4. Final decision: the signal comes first, then the route
    if signal in STOP_SIGNALS or signal not in signal_rules:
        print(f"FINAL DECISION: Wait at the signal. When it is safe, drive towards {nxt}.")
    else:
        print(f"FINAL DECISION: Proceed towards {nxt} following the route above.")

# Ask the agent what to do in the current situation, showing the UCS search step by step
decide_action(current_state, verbose_search=True)


=== Autonomous Trailer Decision ===
Location: Nairobi | Destination: Kisumu | Signal: Green
Signal rule: Go: it is safe to proceed.
Step 1: frontier = [Nairobi (g=0)]
Step 2: frontier = [Nairobi-Naivasha (g=91), Nairobi-Narok (g=143)]
Step 3: frontier = [Nairobi-Narok (g=143), Nairobi-Naivasha-Nakuru (g=162)]
Step 4: frontier = [Nairobi-Naivasha-Nakuru (g=162), Nairobi-Narok-Bomet (g=224)]
Step 5: frontier = [Nairobi-Naivasha-Nakuru-Molo (g=213), Nairobi-Narok-Bomet (g=224), Nairobi-Naivasha-Nakuru-Eldoret (g=318)]
Step 6: frontier = [Nairobi-Narok-Bomet (g=224), Nairobi-Naivasha-Nakuru-Molo-Kericho (g=277), Nairobi-Naivasha-Nakuru-Eldoret (g=318)]
Step 7: frontier = [Nairobi-Naivasha-Nakuru-Molo-Kericho (g=277), Nairobi-Narok-Bomet-Kericho (g=300), Nairobi-Narok-Bomet-Kisii (g=306), Nairobi-Naivasha-Nakuru-Eldoret (g=318)]
Step 8: frontier = [Nairobi-Narok-Bomet-Kericho (g=300), Nairobi-Narok-Bomet-Kisii (g=306), Nairobi-Naivasha-Nakuru-Eldoret (g=318), Nairobi-Naivasha-Nakuru-Molo-K

## Step 8: Querying the Knowledge Base

We now ask the agent different questions to see how it reasons with the knowledge it has. The most important test is **re-planning**: when we tell the agent there is an accident, it should find a new route on its own.

In [8]:
# Query 1: the trailer is at a red light in Nakuru
decide_action({"location": "Nakuru", "destination": "Kisumu", "signal": "Red"})

# Query 2: the optimal route with the roads as they are now
path, cost = ucs("Nairobi", "Kisumu")
print(f"\nQuery: best route Nairobi -> Kisumu = {' -> '.join(path)} ({cost} km)")


=== Autonomous Trailer Decision ===
Location: Nakuru | Destination: Kisumu | Signal: Red
Signal rule: Stop: it is unsafe to move.
Best route: Nakuru -> Molo -> Kericho -> Kisumu (199 km)
Next road: Nakuru -> Molo is Clear: Drive normally within the speed limit.
Ahead: Molo -> Kericho has Fog: Slow down, switch on fog lights, keep a long following distance.
FINAL DECISION: Wait at the signal. When it is safe, drive towards Molo.

Query: best route Nairobi -> Kisumu = Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (361 km)


In [9]:
# Query 3: an accident is reported between Molo and Kericho. The agent re-plans.
report_condition("Molo", "Kericho", "Accident")
decide_action({"location": "Nairobi", "destination": "Kisumu", "signal": "Green"})

# Query 4: the trailer has already reached Nakuru when it hears about the accident
decide_action({"location": "Nakuru", "destination": "Kisumu", "signal": "Green"})

Added fact: Condition(Molo, Kericho, Accident) (road is now CLOSED)

=== Autonomous Trailer Decision ===
Location: Nairobi | Destination: Kisumu | Signal: Green
Signal rule: Go: it is safe to proceed.
Best route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (384 km)
Next road: Nairobi -> Narok is Clear: Drive normally within the speed limit.
FINAL DECISION: Proceed towards Narok following the route above.

=== Autonomous Trailer Decision ===
Location: Nakuru | Destination: Kisumu | Signal: Green
Signal rule: Go: it is safe to proceed.
Best route: Nakuru -> Eldoret -> Kapsabet -> Kisumu (299 km)
Next road: Nakuru -> Eldoret is Weighbridge: Stop at the weighbridge to have the load weighed.
FINAL DECISION: Proceed towards Eldoret following the route above.


In [10]:
# Query 5: the accident is cleared, so the road is Clear again
report_condition("Molo", "Kericho", "Clear")
decide_action({"location": "Nairobi", "destination": "Kisumu", "signal": "Yellow"})

Added fact: Condition(Molo, Kericho, Clear)

=== Autonomous Trailer Decision ===
Location: Nairobi | Destination: Kisumu | Signal: Yellow
Signal rule: Slow down: prepare to stop.
Best route: Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (361 km)
Next road: Nairobi -> Naivasha is Clear: Drive normally within the speed limit.
FINAL DECISION: Proceed towards Naivasha following the route above.


## Step 9: Debugging the Knowledge Base

We test the agent with **unusual or faulty inputs** to check that the KB and the reasoning behave correctly.

In [11]:
print("--- Test 1: signals (known, new and unknown) ---")
for s in ["Red", "Yellow", "Green", "Pedestrian Crossing", "Blue", "Purple", "None"]:
    print(f"Signal: {s:20} -> {signal_rules.get(normalise(s), UNKNOWN_SIGNAL_ACTION)}")

print("\n--- Test 2: messy input (capitals and spaces) ---")
decide_action({"location": "  nAIROBI ", "destination": "kisumu", "signal": "green"})

print("\n--- Test 3: a town that is not in the KB ---")
decide_action({"location": "Nairobi", "destination": "Mombasa", "signal": "Green"})

print("\n--- Test 4: already at the destination ---")
decide_action({"location": "Kisumu", "destination": "Kisumu", "signal": "Green"})

print("\n--- Test 5: invalid facts are rejected ---")
add_road("Nakuru", "Kisumu", -50)
report_condition("Nairobi", "Kisumu", "Fog")       # no such road
report_condition("Nakuru", "Molo", "Tornado")      # unknown condition

--- Test 1: signals (known, new and unknown) ---
Signal: Red                  -> Stop: it is unsafe to move.
Signal: Yellow               -> Slow down: prepare to stop.
Signal: Green                -> Go: it is safe to proceed.
Signal: Pedestrian Crossing  -> Slow down and give way to pedestrians.
Signal: Blue                 -> Unknown signal: STOP and wait until it is safe (safety first).
Signal: Purple               -> Unknown signal: STOP and wait until it is safe (safety first).
Signal: None                 -> Unknown signal: STOP and wait until it is safe (safety first).

--- Test 2: messy input (capitals and spaces) ---

=== Autonomous Trailer Decision ===
Location: Nairobi | Destination: Kisumu | Signal: Green
Signal rule: Go: it is safe to proceed.
Best route: Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (361 km)
Next road: Nairobi -> Naivasha is Clear: Drive normally within the speed limit.
FINAL DECISION: Proceed towards Naivasha following the route above.

---

In [12]:
print("--- Test 6: every road into Kisumu is closed ---")
into_kisumu = [town for town, nb in roads.items() if "Kisumu" in nb]
for town in into_kisumu:
    report_condition(town, "Kisumu", "Flooded")
decide_action({"location": "Nairobi", "destination": "Kisumu", "signal": "Green"})

# Clean up: reopen the roads
for town in into_kisumu:
    report_condition(town, "Kisumu", "Clear")
print("\nAfter reopening:", ucs("Nairobi", "Kisumu"))

--- Test 6: every road into Kisumu is closed ---
Added fact: Condition(Kericho, Kisumu, Flooded) (road is now CLOSED)
Added fact: Condition(Kisii, Kisumu, Flooded) (road is now CLOSED)
Added fact: Condition(Kapsabet, Kisumu, Flooded) (road is now CLOSED)
Added fact: Condition(Kakamega, Kisumu, Flooded) (road is now CLOSED)

=== Autonomous Trailer Decision ===
Location: Nairobi | Destination: Kisumu | Signal: Green
Signal rule: Go: it is safe to proceed.
FINAL DECISION: NO open route to the destination. STOP and wait for roads to reopen.
Added fact: Condition(Kericho, Kisumu, Clear)
Added fact: Condition(Kisii, Kisumu, Clear)
Added fact: Condition(Kapsabet, Kisumu, Clear)
Added fact: Condition(Kakamega, Kisumu, Clear)

After reopening: (['Nairobi', 'Naivasha', 'Nakuru', 'Molo', 'Kericho', 'Kisumu'], 361)


**What the debugging showed:**

- **Unknown signals** (Blue, Purple, None) now return a clear **STOP** instead of the vague *"proceed with caution or stop"* from the traffic light example. When a safety agent is unsure, it should choose the safest action.
- **Messy input** such as `"  nAIROBI "` would not match `"Nairobi"` in the KB with a plain dictionary lookup. `normalise()` removes spaces and fixes capitals before every lookup, so the test passes.
- **Unknown towns** (Mombasa) and **impossible facts** (negative distances, roads that don't exist, unknown conditions) are rejected with a message instead of crashing the program or corrupting the KB.
- When **no route exists**, the agent stops instead of driving on blindly.

### Optional: try the agent yourself

Set `INTERACTIVE = True` and run this cell to type in your own situation. It is off by default so that **Run all** finishes without waiting for input.

In [13]:
INTERACTIVE = False

if INTERACTIVE:
    loc  = input("Where is the trailer now? ")
    dest = input("Where is it going? ")
    sig  = input("What signal does it see? ")
    decide_action({"location": loc, "destination": dest, "signal": sig}, verbose_search=True)
else:
    print("Interactive mode is off. Set INTERACTIVE = True to try your own inputs.")

Interactive mode is off. Set INTERACTIVE = True to try your own inputs.


## Step 10: Reflection

### Challenges we faced

- **Choosing which towns and roads to include.** The class map has many towns, and at first we were not sure which ones mattered. We had to apply abstraction: we kept only the main junction towns between Nairobi and Kisumu and left out towns like Nyeri, Meru and Thika that lead away from Kisumu. Getting this right mattered, because everything else (the graph, the search and the knowledge base) was built on it.
- **Understanding Uniform Cost Search.** It was confusing why UCS does not stop the first time it reaches Kisumu. Tracing the frontier step by step helped us see that a cheaper path to Kisumu might still be waiting in the frontier, so the search only stops when the Kisumu path is the cheapest one left. That is what makes the answer optimal.
- **Turning a graph into a knowledge base.** In Practical 3 the graph was only used for searching. Here we had to think about it as **facts** (`Road(x, y, d)`) and add **rules** the agent can reason with, for example that an accident closes a road and that a closed road means the agent must find another route.

### What we learned

- **The shortest route is not always the one that looks shortest.** Looking at the map, we expected the Narok road (B3) to be the quicker way to Kisumu because it looks more direct. After calculating the path costs, the middle route through Naivasha, Nakuru, Molo and Kericho came out shortest at 361 km, 23 km less than the best Narok route (384 km). This showed us why an agent needs a search algorithm instead of guessing.

### Recommended improvements

- **Live traffic and weather data.** Right now we report fog, rain and accidents by hand. With APIs such as Google Maps/OpenRouteService for traffic and OpenWeatherMap for weather, the agent could update its knowledge base automatically.
- **Use travel time as the path cost.** At the moment fog or road works only produce a warning. If the cost were time instead of distance, a foggy or congested road would become "more expensive", and the agent might choose a different route.
- **Trailer-specific rules.** A real trailer has extra limits: weight limits on some bridges and roads, and restrictions on heavy vehicles driving at night. Adding these rules would make the agent's decisions more realistic.

## Evidence of Group Work

Our group (Laureen, Moses and Vanessa) worked together on this assignment. When we could meet physically, we sat together on one shared laptop; when we could not, we met online and shared the screen. We went through each step as a team

Our team photo is uploaded to this repository as `team_photo.jpg`.